In [3]:
import pandas as pd
import string, json, re

In [4]:
path = "../../output/snc/cctv.xlsx" 
df = pd.read_excel(path, sheet_name="cctv", keep_default_na=False)

base_cols = df[["workorder_id", "filename"]]
meta_cols = df[["station", "datetime", "comment_recommendation", "performed_by", "verified_by", "remark"]]

cctv_checklist = []

# insert the column name that start with cctvlist
for col in df.columns:
    if any(col.startswith(prefix) for prefix in ["safety_and_preparation", "cables", "cleanliness", "cctv", "dvr", "ptz_cameras", "see_eyes"]):
        cctv_checklist.append(col)


# cctv_status is df column that does not exists in base_cols and meta_cols and cctv_checklist
cctv_status = [col for col in df.columns if col not in base_cols.columns and col not in meta_cols.columns and col not in cctv_checklist]

## Flatten into JSON

In [5]:
def to_float(value):
    """
    Converts a value to float. 
    Returns 0.0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return float(clean_val)
    except (ValueError, TypeError):
        return None

def to_int(value):
    """
    Converts a value to int. 
    Returns 0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return int(clean_val)
    except (ValueError, TypeError):
        return None
    
def to_bool(value):
    """
    Converts a value to boolean.
    Returns None if the value is None or empty.
    """
    if pd.isna(value):
        return None
    
    str_val = str(value).strip()
    
    if str_val.upper() == "N/A":
        return "N/A"
    
    if str_val == "":
        return None
    
    str_val = str(value).lower()
    
    if str_val in ['true', '1', 'yes', 'pass']:
        return True
    elif str_val in ['false', '0', 'no', 'notpass']:
        return False
    else:
        return None

def process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns):
    """
    Flattens all columns in the dataframe into a simple key:value JSON structure.
    """
    df = df.rename(columns=rename_columns)
    exclude_from_data = ["workorder_id", "filename"] + exclude_cols

    for _, row in df.iterrows():
        fname = row.get("filename", "unknown")
        wo_id = str(row.get("workorder_id", ""))
        
        if not fname or fname == "nan":
            continue

        if fname not in final_json:
            final_json[fname] = {
                "workorder_id": wo_id,
                "data": {}
            }

        row_flattened_data = {}
        for col in df.columns:
            if col in exclude_from_data:
                continue
            
            value = row[col]
            
            if any(num_col in col for num_col in float_columns):
                row_flattened_data[col] = to_float(value)
            elif any(num_col in col for num_col in int_columns):
                row_flattened_data[col] = to_int(value)
            elif any(bool_col in col for bool_col in bool_columns):
                row_flattened_data[col] = to_bool(value)
            else:
                row_flattened_data[col] = value if pd.notna(value) else None
                
        final_json[fname]["data"].update(row_flattened_data)

    return final_json

### CCTV

In [6]:
df_cctv = df.copy()

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
exclude_cols = []
rename_columns = {}

bool_columns += cctv_checklist + cctv_status

process_flattened_sheet(df_cctv, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/snc/response_cctv.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/snc/response_cctv.xlsx
